# 05 — Segmentation & Inventory

Trains the KMeans persona model and the inventory recommender
in one pass so we can compare the ABC distribution and the
top-reorder SKUs side by side.

In [ ]:
import logging
import pandas as pd
import matplotlib.pyplot as plt

from neuralretail.data.ingest import load_raw
from neuralretail.data.clean import clean
from neuralretail.features.rfm import compute_rfm
from neuralretail.models.segmentation import train as seg_train
from neuralretail.models.inventory import train as inv_train

logging.getLogger('neuralretail').setLevel(logging.WARNING)


## Train segmentation

In [ ]:
raw = load_raw()
cleaned, _ = clean(raw)
rfm = compute_rfm(cleaned)
seg = seg_train(rfm, k_min=4, k_max=8, run_name='nb_seg')
print(f'k = {seg.k} | silhouette = {seg.metrics["silhouette"]:.4f}')
seg.summary


## Recency × Monetary scatter coloured by cluster

Each point is one customer; the persona name comes from the
post-training cluster → persona mapping.

In [ ]:
scored = rfm.copy()
scored['cluster'] = seg.labels
scored['persona'] = scored['cluster'].map(seg.persona_map)
fig, ax = plt.subplots(figsize=(8, 6))
for persona, sub in scored.groupby('persona'):
    ax.scatter(sub['Recency'], sub['Monetary'], label=persona, alpha=0.6, s=14)
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Monetary (GBP)')
ax.set_yscale('log')
ax.set_title(f'Customer segments (k={seg.k}, silhouette={seg.metrics["silhouette"]:.3f})')
ax.legend()
plt.tight_layout()
plt.show()


## Train inventory and inspect ABC distribution

In [ ]:
inv = inv_train(cleaned, run_name='nb_inv')
print('inventory rows:', len(inv.table))
print('metrics:', {k: round(v, 4) if isinstance(v, float) else v for k, v in inv.metrics.items()})
abc_counts = inv.table['ABC'].value_counts()
ax = abc_counts.plot.pie(autopct='%1.0f%%', colors=['#2a9d8f', '#e9c46a', '#f4a261'])
ax.set_title('ABC class distribution')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## Top 10 reorder candidates

In [ ]:
top = inv.table.sort_values('Revenue', ascending=False).head(10)
top[['StockCode', 'Description', 'ABC', 'UnitsSold', 'Revenue', 'EOQ', 'IsDeadStock']]
